# Modelo de Probabilidad of Default 

Desarrollo del modelo de Probablidad of Default.

1. Primer enfoque básico: En este primer enfoque concentraremos en construir el modelo sin modificaciones, utilizando las variables del EDA y del VIF, sin contar con los logaritmos. 
2. Enfoque tuneado. 

### Importamos librerias 

In [56]:
import polars as pl
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from lightgbm import LGBMClassifier

from sklearn.model_selection import train_test_split
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import  RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    log_loss,
    brier_score_loss,
)

from scipy import stats

pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_str_lengths(60)
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.titleweight"] = "bold"

SEED = 42
CV_SAMPLE_SIZE = 200_000

### Carga y taxonomia de variables.

Separamos nuestras variables`

In [57]:
Xtr_pl = pl.read_parquet("../data/processed/X_train.parquet")

TARGET = "y"

LEAKAGE_VARS = ['grade']

CAT  = [c for c, dt in Xtr_pl.schema.items() if dt == pl.String]
NUM  = [c for c, dt in Xtr_pl.schema.items() if dt.is_numeric()]

LOG_VARS   = [c for c in NUM if c.startswith("log_")]    # 3 logs creados en 02
FLAG_VARS  = [c for c in NUM if c.startswith("flag_")]   # 5 flags de imputación
NUM_BASE   = [c for c in NUM if c not in LOG_VARS + FLAG_VARS]  # 29 originales
RAW_OF_LOG = [c.removeprefix("log_") for c in LOG_VARS]  # variables que tienen su log
VIF_DROP   = ["monthly_payment"]                         # VIF > 10 (colineal con funded_amnt)

# VARS = NUM + CAT

X_train = Xtr_pl.to_pandas()
X_test  = pl.read_parquet("../data/processed/X_test.parquet").to_pandas()
y_train = pl.read_parquet("../data/processed/y_train.parquet")["y"].to_numpy()
y_test  = pl.read_parquet("../data/processed/y_test.parquet")["y"].to_numpy()

print(f"num_base={len(NUM_BASE)} | logs={len(LOG_VARS)} | flags={len(FLAG_VARS)} | cat={len(CAT)}")

num_base=29 | logs=3 | flags=5 | cat=10


Preparamos las variables categóricas para LightGBM. Esto permite identificar y mantener una codificación consistente entre ambos conjuntos.

Además, evita tratar las categorías como números con un orden artificial. Las categorías no observadas durante el entrenamiento pueden convertirse en valores faltantes (NaN) al aplicar esta transformación.

In [58]:
CATEGORIES = {c: pd.Categorical(X_train[c]).categories for c in CAT}

def to_lgbm(df):
    out = df.copy()
    for c in CAT:
        out[c] = pd.Categorical(out[c], categories=CATEGORIES[c])
    return out

X_train_lgb, X_test_lgb = to_lgbm(X_train), to_lgbm(X_test)

1. Debido a que contamos con mas de ~2M de datos, seleccionaremos una muestra representativa y estratificado de 200_000 prestamos del conjunto de entrenamiento (X_train). La estratificación permite conservar aproximadamente la proporción original de incumplimientos (defaults). 

2. La muestra se dividió en dos subconjuntos, con el fin de evaluar la capacidad de generalizacion con datos cuyas etiquetas no se utilizaron para ajustar los parámetros del modelo. 

    - 80 % (160,000 préstamos): entrenamiento de los modelos.
    - 20 % (40,000 préstamos): validación y comparación del rendimiento.


In [ ]:
# generamos los indices de las filas para nuestra muestra en el conjunto de entrenamiento
# muestra de 200_000
idx_dev, _ = train_test_split(
    np.arange(len(y_train)),
    train_size=200_000,
    stratify=y_train,
    random_state=SEED
)

# construimos los datos para CV
X_dev = X_train.iloc[idx_dev]
X_dev_lgb = X_train_lgb.iloc[idx_dev]
y_dev = y_train[idx_dev]

# División: 80 % entrenamiento y 20 % validación
idx_fit, idx_valid = train_test_split(
    np.arange(len(y_dev)),
    test_size=0.20,
    stratify=y_dev,
    random_state=SEED
)

# Conjuntos de entrenamiento
X_fit = X_dev.iloc[idx_fit]
X_fit_lgb = X_dev_lgb.iloc[idx_fit]
y_fit = y_dev[idx_fit]

# Conjuntos de validación
X_valid = X_dev.iloc[idx_valid]
X_valid_lgb = X_dev_lgb.iloc[idx_valid]
y_valid = y_dev[idx_valid]

# configguramos nuestra validación cruzada de 5 folds. 
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

## Construcción de Modelos básicos

### Regresión logística (LR)

Se definieron cuatro variantes para comparar el efecto de las transformaciones logarítmicas y la exclusión de monthly_payment, variable identificada con VIF elevado.

- `C=1.0:` establece una regularización moderada por defecto. En regresión logística, valores menores implican mayor regularización; este parámetro se ajustará posteriormente.
- `solver="lbfgs":` optimizador adecuado para regresión logística multiclase o binaria con regularización L2, especialmente con conjuntos de datos de tamaño considerable.
- `max_iter=1000:` permite suficientes iteraciones para alcanzar la convergencia.
- `class_weight=None:` conserva el peso natural de las clases, evitando reponderar artificialmente los incumplimientos y alterar las probabilidades estimadas.
- `StandardScaler:`estandariza las variables numéricas continuas para facilitar la optimización y evitar que las diferencias de escala dominen el ajuste.
- `OneHotEncoder(handle_unknown="ignore", min_frequency=0.005):` codifica las variables categóricas y agrupa las categorías con frecuencia inferior al 0.5 %, reduciendo la cantidad de columnas y el impacto de niveles poco frecuentes.
- `Variables binarias (FLAG_VARS):` se mantienen sin escalar, ya que representan indicadores de imputación.

In [60]:
def swap_raw_for_log(cols):
    """Sustituye cada variable cruda por su log (si existe)."""
    return [f"log_{c}" if c in RAW_OF_LOG else c for c in cols]

NUM_NOVIF = [c for c in NUM_BASE if c not in VIF_DROP]

CONT_SETS = {
    "LR_A (crudas, todas)" : NUM_BASE,
    "LR_B (crudas, sin VIF)" : NUM_NOVIF,
    "LR_C (logs, todas)": swap_raw_for_log(NUM_BASE),
    "LR_D (logs, sin VIF)": swap_raw_for_log(NUM_NOVIF) 
}

def make_lr(cont_cols):
    """Pipeline LR: el preprocesado se ajusta dentro de cada fold (sin fuga).
    ColumnTransformer descarta (remainder='drop') toda columna no listada, así
    cada variante usa exactamente las variables que declara."""
    prep = ColumnTransformer([
        ("num",  StandardScaler(), cont_cols),          # LR necesita escala común
        ("flag", "passthrough",    FLAG_VARS),          # binarias: sin escalar
        ("cat",  OneHotEncoder(handle_unknown="ignore", # nivel nuevo -> todo ceros
                               min_frequency=0.005,     # niveles raros -> 1 grupo
                               sparse_output=True), CAT),
    ])
    
    model = LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
        class_weight=None,       # NO reponderar: necesitamos PD calibrada
        random_state=SEED)

    return Pipeline([("prep", prep), ("model", model)])

### LightGBM (LGBM)

Se estableció un modelo base de clasificación binaria con n_estimators=100, es decir, 100 árboles de boosting.

- `objective="binary":` define la tarea como clasificación binaria para estimar la probabilidad de incumplimiento.
- `n_estimators=100:` proporciona un punto de partida sencillo para evaluar el rendimiento antes de ajustar la complejidad del modelo.
- `random_state=SEED:` permite reproducir los resultados bajo las mismas condiciones.
- `n_jobs=-1:` utiliza los núcleos de CPU disponibles para acelerar el entrenamiento.
- `verbosity=-1:`reduce los mensajes de salida durante la ejecución.

LightGBM utiliza las variables categóricas en formato category de pandas, sin necesidad de escalado ni One-Hot Encoding. No se activaron is_unbalance ni scale_pos_weight, para evitar introducir una ponderación artificial de las clases que pueda afectar la interpretación de las probabilidades.


In [61]:
# LGBM sin pipeline: no necesita escalado ni one-hot. Sin logs (invariancia).
FEATURES_LGBM = NUM_BASE + FLAG_VARS + CAT

lgbm_base = LGBMClassifier(
    objective="binary",
    n_estimators=100, 
    # sin is_unbalance / scale_pos_weight, por la misma razón de calibración
    random_state=SEED, 
    n_jobs=-1, 
    verbosity=-1
)

### Métricas de evaluación de los modelos

Se utilizaron cuatro indicadores para evaluar la capacidad predictiva y el comportamiento agregado de las probabilidades de incumplimiento (PD).

- **ROC-AUC:** mide la capacidad del modelo para distinguir entre préstamos que incumplen y los que no, independientemente de un umbral de clasificación. Se prefiere frente a accuracy porque esta última puede resultar engañosa cuando existe desbalance entre clases.

- **LogLoss:** evalúa la calidad de las probabilidades estimadas y penaliza especialmente las predicciones muy seguras que resultan incorrectas. Es fundamental porque el objetivo no es únicamente clasificar préstamos, sino estimar PD para el cálculo de pérdidas crediticias esperadas (ECL).

- **PD media:** representa el promedio de las probabilidades de incumplimiento estimadas. Permite contrastar el nivel de riesgo promedio predicho con la tasa real de incumplimiento de la muestra.

- **SumPD/Defaults:** relaciona la suma de las PD estimadas con el número observado de incumplimientos. Un resultado cercano a 1 indica concordancia agregada entre los defaults esperados y los observados, aunque no garantiza una calibración correcta en todos los segmentos.

¿Por qué no utilizar únicamente métricas clásicas?

Métricas como ``accuracy, precisión, recall y F1-score`` requieren establecer un umbral para convertir probabilidades en clases. Esto puede ser útil para decisiones operativas, pero pierde información sobre la magnitud de las probabilidades.

Para este proyecto,**AUC y LogLoss** permiten evaluar discriminación y calidad probabilística, mientras que PD media y SumPD/Defaults aportan una comprobación agregada relevante para el riesgo crediticio.

Estas métricas evalúan la PD; por sí solas no determinan la calidad final de la ECL, que también depende de la pérdida dado el incumplimiento (LGD) y la exposición al incumplimiento (EAD).

In [78]:
def metricas_pd(y, p, nombre):
    auc = roc_auc_score(y, p)
    ll = log_loss(y, p, labels=[0, 1])

    pd_media = np.mean(p)
    defaults_reales = np.sum(y)

    ratio_pd_defaults = (
        np.sum(p) / defaults_reales
        if defaults_reales > 0
        else np.nan
    )

    return {
        "modelo": nombre,
        "AUC": auc,
        "LogLoss": ll,
        "PD_media": pd_media,
        "SumPD/Defaults": ratio_pd_defaults,
    }


In [79]:
resultados = []

# Modelo para cada grupo de RL
for name, cols in CONT_SETS.items():
    model = make_lr(cols)
    model.fit(X_fit,y_fit)

    #validamos 
    pd_pred = model.predict_proba(X_valid)[:,1]

    resultados.append(metricas_pd(y_valid, pd_pred, name))

#modelo para LGB
lgbm_base.fit(X_fit_lgb[FEATURES_LGBM], y_fit)

#validamos
pd_pred_lgbm = lgbm_base.predict_proba(X_valid_lgb[FEATURES_LGBM])[:,1]

resultados.append(metricas_pd(y_valid, pd_pred_lgbm, "LGBM_base"))

# Tabla comparativa
tabla_modelos = (
    pd.DataFrame(resultados)
    .set_index("modelo")
    .sort_values("LogLoss")
    .round(4)
)

tabla_modelos

,AUC,LogLoss,PD_media,SumPD/Defaults
modelo,,,,
LGBM_base,0.7530,0.3389,0.1312,1.0038
"LR_D (logs, sin VIF)",0.7255,0.3503,0.1316,1.0067
"LR_C (logs, todas)",0.7253,0.3504,0.1316,1.0067
"LR_B (crudas, sin VIF)",0.7251,0.3504,0.1315,1.0059
"LR_A (crudas, todas)",0.7251,0.3504,0.1315,1.0059


- LightGBM obtuvo el mejor rendimiento, con un AUC de 0.7530 y un LogLoss de 0.3389. Esto indica una mayor capacidad para distinguir entre préstamos que incumplen y los que no, así como una mejor calidad de las probabilidades estimadas respecto a las variantes LR evaluadas.

- La regresión logística presentó resultados similares entre sus cuatro variantes. La mejor fue LR_D (logs, sin VIF), aunque su mejora frente a las demás variantes fue mínima. Esto sugiere que las transformaciones logarítmicas y la exclusión de monthly_payment aportaron poca mejora en esta configuración base.

In [ ]:
# preprocessor_lr = ColumnTransformer(
#     transformers=[
#         ("num",StandardScaler(),NUM),
#         ("cat",OneHotEncoder(handle_unknown="ignore",sparse_output=True),CAT)
#     ]
# )

# model_lr = LogisticRegression(
#     max_iter=1000,
#     class_weight=None,
#     random_state=RANDOM_STATE
# )

# logistic_pipeline = Pipeline([
#     ("preprocessor", preprocessor_lr),
#     ("model", model_lr)
# ])

# param_grid_lr = {
#     "model__C": [0.01, 0.1, 1, 10],
#     "model__solver": ["lbfgs","liblinear"]
# }

# grid_lr = GridSearchCV(
#     estimator=logistic_pipeline,
#     param_grid=param_grid_lr,
#     cv=cv,
#     scoring="roc_auc",
#     n_jobs=-1,
#     verbose=2,
#     return_train_score=False
# )

In [ ]:
# grid_lr.fit( X_grid,y_grid)
# print("Mejores parámetros:")
# print(grid_lr.best_params_)

# print(f"\nMejor ROC-AUC CV: {grid_lr.best_score_:.4f}")

Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV] END .................model__C=0.01, model__solver=lbfgs; total time=   9.7s
[CV] END .................model__C=0.01, model__solver=lbfgs; total time=  10.5s
[CV] END .................model__C=0.01, model__solver=lbfgs; total time=  10.0s
[CV] END .................model__C=0.01, model__solver=lbfgs; total time=  10.8s
[CV] END .................model__C=0.01, model__solver=lbfgs; total time=  11.1s
[CV] END .............model__C=0.01, model__solver=liblinear; total time=  15.1s
[CV] END .............model__C=0.01, model__solver=liblinear; total time=  14.7s
[CV] END .............model__C=0.01, model__solver=liblinear; total time=  16.6s
[CV] END ..................model__C=0.1, model__solver=lbfgs; total time=   9.7s
[CV] END ..................model__C=0.1, model__solver=lbfgs; total time=   9.0s
[CV] END ..................model__C=0.1, model__solver=lbfgs; total time=  11.7s
[CV] END ..................model__C=0.1, model__s

In [ ]:
# best_lr = grid_lr.best_estimator_

# best_lr.fit(X_train, y_train)

# y_pred = best_lr.predict(X_test)
# y_proba = best_lr.predict_proba(X_test)[:, 1]

# from sklearn.metrics import (
#     accuracy_score,
#     recall_score,
#     precision_score,
#     f1_score,
#     roc_auc_score,
#     confusion_matrix
# )

# accuracy = accuracy_score(y_test, y_pred)
# recall = recall_score(y_test, y_pred)
# precision = precision_score(y_test, y_pred)
# f1 = f1_score(y_test, y_pred)
# roc_auc = roc_auc_score(y_test, y_proba)

# print(f"Accuracy:  {accuracy:.4f}")
# print(f"Recall:    {recall:.4f}")
# print(f"Precision: {precision:.4f}")
# print(f"F1-Score:  {f1:.4f}")
# print(f"ROC-AUC:   {roc_auc:.4f}")

# print("\nMatriz de confusión:")
# print(confusion_matrix(y_test, y_pred))

Accuracy:  0.8690
Recall:    0.0174
Precision: 0.4751
F1-Score:  0.0335
ROC-AUC:   0.7273

Matriz de confusión:
[[370912   1074]
 [ 54971    972]]


In [ ]:
# for col in CAT:
#     categories = X_train[col].astype("category").cat.categories

#     X_train[col] = pd.Categorical(
#         X_train[col],
#         categories=categories
#     )

#     X_grid[col] = pd.Categorical(
#         X_grid[col],
#         categories=categories
#     )

#     X_test[col] = pd.Categorical(
#         X_test[col],
#         categories=categories
#     )

In [ ]:
# from sklearn.model_selection import RandomizedSearchCV
# from scipy.stats import randint, uniform

# param_dist = {
#     "num_leaves": randint(15, 127),
#     "min_child_samples": randint(200, 2000),
#     "reg_lambda": uniform(0, 20),
#     "colsample_bytree": uniform(0.5, 0.5),
#     "subsample": uniform(0.6, 0.4),
# }

# lgbm = LGBMClassifier(
#     objective="binary",
#     n_estimators=400,
#     learning_rate=0.05,
#     subsample_freq=1,
#     random_state=RANDOM_STATE,
#     n_jobs=-1,
#     verbosity=-1
# )

# search_lgbm = RandomizedSearchCV(
#     estimator=lgbm,
#     param_distributions=param_dist,
#     n_iter=20,
#     cv=cv,
#     scoring={
#         "auc": "roc_auc",
#         "logloss": "neg_log_loss"
#     },
#     refit="logloss",
#     n_jobs=1,
#     random_state=RANDOM_STATE,
#     verbose=2,
#     return_train_score=False
# )

# search_lgbm.fit(
#     X_grid,
#     y_grid,
#     categorical_feature=CAT
# )

# print("Best params:")
# print(search_lgbm.best_params_)

# print("Best CV Log-loss:")
# print(-search_lgbm.best_score_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
[CV] END learning_rate=0.01, n_estimators=100, num_leaves=15; total time=   2.4s
[CV] END learning_rate=0.01, n_estimators=100, num_leaves=15; total time=   2.0s
[CV] END learning_rate=0.01, n_estimators=100, num_leaves=15; total time=   1.9s
[CV] END learning_rate=0.01, n_estimators=100, num_leaves=15; total time=   2.2s
[CV] END learning_rate=0.01, n_estimators=100, num_leaves=15; total time=   1.7s
[CV] END learning_rate=0.01, n_estimators=100, num_leaves=31; total time=   2.2s
[CV] END learning_rate=0.01, n_estimators=100, num_leaves=31; total time=   2.2s
[CV] END learning_rate=0.01, n_estimators=100, num_leaves=31; total time=   2.8s
[CV] END learning_rate=0.01, n_estimators=100, num_leaves=31; total time=   2.8s
[CV] END learning_rate=0.01, n_estimators=100, num_leaves=31; total time=   2.1s
[CV] END learning_rate=0.01, n_estimators=100, num_leaves=63; total time=   3.5s
[CV] END learning_rate=0.01, n_estimators=100, 

KeyboardInterrupt: 

In [ ]:
# for col in CAT:
#     categories = X_train[col].astype("category").cat.categories

#     X_train[col] = pd.Categorical(
#         X_train[col],
#         categories=categories
#     )

In [ ]:
# best_lgbm = grid_lgbm.best_estimator_

# best_lgbm.fit(X_train,y_train,categorical_feature=CAT)

# pd_lr = best_lr.predict_proba(X_test)[:, 1]
# pd_lgbm = best_lgbm.predict_proba(X_test)[:, 1]

In [ ]:
# from scipy.stats import ks_2samp
# from sklearn.metrics import log_loss, brier_score_loss
# from sklearn.metrics import roc_auc_score

# def evaluar_pd(y, pd_pred, nombre):
#     ks = ks_2samp(
#         pd_pred[y == 1],
#         pd_pred[y == 0]
#     ).statistic

#     pd_sum = pd_pred.sum()
#     defaults = y.sum()

#     df_cal = pd.DataFrame({
#         "y": y,
#         "pd": pd_pred
#     })

#     df_cal["decil"] = pd.qcut(
#         df_cal["pd"],
#         10,
#         duplicates="drop"
#     )

#     calibracion = (
#         df_cal
#         .groupby("decil", observed=True)
#         .agg(
#             PD_predicha=("pd", "mean"),
#             Default_real=("y", "mean"),
#             N=("y", "size")
#         )
#     )

#     print(f"\n{'=' * 30}")
#     print(nombre)
#     print(f"{'=' * 30}")
#     print(f"Log-loss:       {log_loss(y, pd_pred):.4f}")
#     print(f"Brier:          {brier_score_loss(y, pd_pred):.4f}")
#     print(f"KS:             {ks:.4f}")
#     print(f"Sum(PD):        {pd_sum:.2f}")
#     print(f"Defaults reales:{defaults}")
#     print(f"Ratio:          {pd_sum / defaults:.4f}")

#     print(f"ROC-AUC:        {roc_auc_score(y, pd_pred):.4f}")
#     return calibracion

In [ ]:
# cal_lr = evaluar_pd(
#     y_test,
#     pd_lr,
#     "Logistic Regression"
# )

# cal_lgbm = evaluar_pd(
#     y_test,
#     pd_lgbm,
#     "LightGBM"
# )


Logistic Regression
Log-loss:       0.6132
Brier:          0.2133
KS:             0.3330
Sum(PD):        189068.16
Defaults reales:55943
Ratio:          3.3797
ROC-AUC:        0.7277

LightGBM
Log-loss:       0.3360
Brier:          0.1018
KS:             0.3824
Sum(PD):        55943.45
Defaults reales:55943
Ratio:          1.0000
ROC-AUC:        0.7599


In [ ]:
# best_lgbm = grid_lgbm.best_estimator_

# best_lgbm.fit(X_train,y_train,categorical_feature=CAT)

# y_pred = best_lgbm.predict(X_test)
# y_proba = best_lgbm.predict_proba(X_test)[:, 1]

# from sklearn.metrics import (
#     accuracy_score,
#     recall_score,
#     precision_score,
#     f1_score,
#     roc_auc_score,
#     confusion_matrix
# )

# accuracy = accuracy_score(y_test, y_pred)
# recall = recall_score(y_test, y_pred)
# precision = precision_score(y_test, y_pred)
# f1 = f1_score(y_test, y_pred)
# roc_auc = roc_auc_score(y_test, y_proba)

# print(f"Accuracy:  {accuracy:.4f}")
# print(f"Recall:    {recall:.4f}")
# print(f"Precision: {precision:.4f}")
# print(f"F1-Score:  {f1:.4f}")
# print(f"ROC-AUC:   {roc_auc:.4f}")

# print("\nMatriz de confusión:")
# print(confusion_matrix(y_test, y_pred))

Accuracy:  0.8696
Recall:    0.0118
Precision: 0.5687
F1-Score:  0.0232
ROC-AUC:   0.7599

Matriz de confusión:
[[371484    502]
 [ 55281    662]]


In [ ]:
# from sklearn.metrics import log_loss, brier_score_loss

# print("Log-loss:", log_loss(y_test, pd_lgbm))
# print("Brier:", brier_score_loss(y_test, pd_lgbm))

In [ ]:
# logistic_pipeline.fit(X_train,y_train)

# y_prob = logistic_pipeline.predict_proba(X_test)[:,1]
# y_pred = logistic_pipeline.predict(X_test)
# feature_names = logistic_pipeline.named_steps["preprocessor"].get_feature_names_out()

# coefficients = logistic_pipeline.named_steps["model"].coef_[0]

# # Coeficientes del modelo
# coef_df = pd.DataFrame({
#     "Variable": feature_names,
#     "Coeficiente": coefficients
# }).sort_values(
#     by="Coeficiente",
#     key=abs,
#     ascending=False
# )

# coef_df.head(20)

KeyboardInterrupt: 

In [ ]:
# marginal_effects = []

# for i, var in enumerate(feature_names):
#     me = np.mean(coefficients[i] * y_prob * (1 - y_prob))
#     marginal_effects.append(me)

# # Crear DataFrame para visualizar
# df_marginal = pd.DataFrame({
#     'Variable': feature_names,
#     'Coeficiente': coefficients,
#     'Efecto_Marginal_Promedio': marginal_effects
# })

# df_marginal.head(20)

,Variable,Coeficiente,Efecto_Marginal_Promedio
0,num__funded_amnt,-0.124362,-0.025832
1,num__interest_rate,0.382460,0.079443
2,num__monthly_payment,0.261421,0.054301
3,num__annual_income,0.006300,0.001309
4,num__dept_paym_income_ratio,0.143545,0.029817
5,num__num_30+_delinq_in_2yrs,0.035474,0.007368
6,num__num_inq_in_6mths,0.067571,0.014036
7,num__mths_since_last_delinq,-0.016160,-0.003357
8,num__num_open_credit_lines,0.067711,0.014065
9,num__num_derogatory_pub_rec,0.074031,0.015377


In [ ]:
# from sklearn.metrics import (
#     roc_auc_score,
#     recall_score,
#     precision_score,
#     f1_score,
#     brier_score_loss
# )

# roc_auc = roc_auc_score(y_test, y_prob)
# recall = recall_score(y_test, y_pred)
# precision = precision_score(y_test, y_pred)
# f1 = f1_score(y_test, y_pred)
# brier = brier_score_loss(y_test, y_prob)

# print(f"ROC-AUC    : {roc_auc:.4f}")
# print(f"Recall     : {recall:.4f}")
# print(f"Precision  : {precision:.4f}")
# print(f"F1         : {f1:.4f}")
# print(f"Brier Score: {brier:.4f}")

ROC-AUC    : 0.7277
Recall     : 0.6736
Precision  : 0.2275
F1         : 0.3401
Brier Score: 0.2133


entropia, umbrall de decisión. 
logloss funcion de perdida. 